### Test notebook for generating bt inputs for CHAZ

use IBTrACS to generate best track (bt) files for each basin and globe


based on read_ibtracs code from clee, which can be found at `taroko/home/clee/lib/python/pygplib3/readbst.py`


IBTrACS can be downloaded directly from here: https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/netcdf/IBTrACS.ALL.v04r01.nc

hosted by:
https://www.ncei.noaa.gov/products/international-best-track-archive 



In [1]:
### TEMPORARY (hopefully) code snippet so that the HTML view of datasets/dataarrays is viewable in darkmode on jupyter lab/VS code
from IPython.display import HTML, display

display(HTML("""
<style>
    .xr-wrap {
        --xr-background-color-row-odd: rgba(100, 100, 100, 0.10) !important;
        --xr-background-color-row-even: rgba(30, 30, 30, 0.10) !important;
    }
</style>
"""))

In [2]:
import Namelist as gv

# gv.ipath+'IBTrACS.ALL.v04r01.nc'


In [3]:
import xarray as xr
import numpy as np
import pandas as pd

#import time
from datetime import datetime

from netCDF4 import Dataset


The following takes about 2 minutes to generate best tracks for all basins and the globe

In [ ]:
BT_FIELDS = [
    'PIwspd', 'PIslp', 'PIwspdMean', 'dPIwspd', 'PIslpMean',
    'UShearMean', 'UShear', 'VShearMean', 'VShear',
    'div200Mean', 'div200', 'T200Mean', 'T200',
    'rh500_300', 'rh500_300Mean', 'rhMean', 'rh',
    'T100Mean', 'T100', 'dThetaEMean', 'dThetaE', 'dThetaEsMean', 'dThetaEs',
    'landmask', 'landmaskMean']

class read_ibtracs_v4_xr(object):
    '''
    From IBTrACS use USA agency (e.g., usa_lat, usa_lon, etc.)
    atl - NHC ATL
    enp - NHC ENP
    wnp,sh,ni - JTWC for the rest: SH, IO, and WPC
    'global' reads all data
    No longer need 'gap', but only keep data at 00,06,12,18z...
    convert lon to 0 to 360 range
    '''

    def __init__(self, ncFileName, basins):
        ds = xr.open_dataset(ncFileName)
 
        sourceN = []
        if 'atl' in basins:
            sourceN.extend(['NA'])
        if 'wnp' in basins:
            sourceN.extend(['WP'])
        if 'sh' in basins:
            sourceN.extend(['SA', 'SP', 'SI'])
        if 'ni' in basins:
            sourceN.extend(['NI'])
        if 'enp' in basins:
            sourceN.extend(['EP'])
        if 'global' in basins:
            sourceN = ['NA', 'WP', 'SA', 'SP', 'SI', 'NI', 'EP']
        sourceN = np.array(sourceN)
        #print(sourceN)
 
        basin_col0 = ds.basin[:, 0].values
        arg = []
        for i in range(sourceN.shape[0]):
            arg.extend(np.argwhere(basin_col0 == sourceN[i].encode('UTF-8')).ravel().tolist())
        arg = np.array(arg)
 
        lon = ds.usa_lon.values[arg]
        lon[lon < 0] = lon[lon < 0] + 360
        nNaN = np.argwhere(np.nanmax(np.array(lon), axis=1) != -9999.).ravel()
        ds1 = ds.sel(storm=arg[nNaN])
 
        hours = ds1.time.dt.hour.values
        arg_a, arg_b = np.where((hours == 0) | (hours == 6) | (hours == 12) | (hours == 18))
 
        storm_ids = np.unique(arg_a)
        n_storms = storm_ids.shape[0]
        d1 = int(hours.shape[1] / 2)
 
        lon = np.full([n_storms, d1], np.nan)
        lat = np.full([n_storms, d1], np.nan)
        wspd = np.full([n_storms, d1], np.nan)
        pres = np.full([n_storms, d1], np.nan)
        dist2land = np.full([n_storms, d1], np.nan)
        trspeed = np.full([n_storms, d1], np.nan)
        trdir = np.full([n_storms, d1], np.nan)
        sdate = np.full([n_storms, d1], np.datetime64('NaT'))
        sdate = np.full([n_storms, d1], np.datetime64('NaT'), dtype=ds1.time.values.dtype)
        status = np.full([n_storms, d1], b'  ', dtype='S2')
 
        for i, iarg in enumerate(storm_ids):
            iarg_b = np.sort(arg_b[arg_a == iarg])
            n = iarg_b.shape[0]
            lon[i, :n] = ds1.usa_lon[iarg, iarg_b].values
            lat[i, :n] = ds1.usa_lat[iarg, iarg_b].values
            wspd[i, :n] = ds1.usa_wind[iarg, iarg_b].values
            pres[i, :n] = ds1.usa_pres[iarg, iarg_b].values
            dist2land[i, :n] = ds1.dist2land[iarg, iarg_b].values
            trspeed[i, :n] = ds1.storm_speed[iarg, iarg_b].values
            trdir[i, :n] = ds1.storm_dir[iarg, iarg_b].values
            sdate[i, :n] = ds1.time[iarg, iarg_b].values
            status[i, :n] = ds1.usa_status[iarg, iarg_b].values
 
        ## convert any leftover IBTrACS fill values to NaN
        for arr in (lon, lat, wspd, pres, dist2land, trspeed, trdir):
            arr[arr == -9999.] = np.nan

        ## account for IBTrACS storing continuous longitudes
        #lon = ((lon + 180) % 360) - 180   # [-180, 180)
        lon = lon % 360 
 
        self.lon = lon
        self.lat = lat
        self.wspd = wspd
        self.pres = pres
        self.dates = sdate
        self.names = ds1.name[storm_ids].values
        self.stormID = ds1.number[storm_ids].values.astype(float)
        self.distland = dist2land
        self.trspeed = trspeed
        self.trdir = trdir
        self.season = ds1.season[storm_ids].values.astype(float)
        #self.basin = ds1.basin[storm_ids].values
        self.basin = ds1.basin[storm_ids, 0].values
        self.status = status

        ##debugging
        ## storms do some wrapping, and IBTrACS reports those as -360 when they
        ## cross the meridian i think? 
        # bad = lon[(lon < -180) | (lon > 180)]
        # print(np.unique(bad)[:20], bad.size)
        # print(np.nanmin(lon), np.nanmax(lon))
        # bad = (lon < -180) | (lon > 180)
        # idx = np.unique(np.where(bad)[0])
        # for i in idx:
        #     print(self.names[i], self.season[i])
        #     print(np.round(lon[i][np.isfinite(lon[i])], 1))

        


## figure out what the correct date is
def _time_to_numeric(dates_2d, clock_start_time=np.datetime64('1800-01-01')):
#def _time_to_numeric(dates_2d, clock_start_time=datetime(1800,1,1,0)):
    """
    Converts a (d1,d2) datetime64 array (NaT = missing) into days since
    `clock_start_time`
 
    TODO: double-check this is the correct start time.
    ### STIL NOT SURE IF THIS IS CORRECT
    """
    days = (dates_2d - clock_start_time) / np.timedelta64(1, 'D')
    return np.where(pd.isna(dates_2d), 0.0, days).astype(float)

def build_bt(ncFileName, basinName, outFileName):
    '''
    reads in ibtracs information, creates best tracks file for each basin and the globe
    generates empty nan dataarrays for all relevant variables for CHAZ that are later calculated with 
    get_predictors. 
    '''

    r = read_ibtracs_v4_xr(ncFileName, [basinName])

    ## build out bt outputs
    StormLon = r.lon.T
    StormLat = r.lat.T
    StormMwspd = r.wspd.T
    StormMslp = r.pres.T
    trSpeed = r.trspeed.T
    trDir = r.trdir.T
    #dist2land = r.distland.T           ## unused in CHAZ
    dates = r.dates.T  # (d1, d2) 
    basin = r.basin

    ## d1, d2 is based on StormLon.shape, but 
    ## lons in 
    d1, d2 = StormLon.shape
    if d2 == 0:
        raise ValueError(f"No storms found for basin '{basinName}'")
 
    StormYYYY = np.full((d1, d2), np.nan)
    StormMM = np.full((d1, d2), np.nan)
    StormDD = np.full((d1, d2), np.nan)
    StormHH = np.full((d1, d2), np.nan)
    StormInitMonth = np.full(d2, np.nan)
 
    for j in range(d2):
        col = pd.DatetimeIndex(dates[:, j])
        valid = ~col.isna()
        if valid.any():
            StormYYYY[valid, j] = col.year[valid]
            StormMM[valid, j] = col.month[valid]
            StormDD[valid, j] = col.day[valid]
            StormHH[valid, j] = col.hour[valid]
            StormInitMonth[j] = col.month[valid][0]
 
    StormId = r.stormID
    ## season just outputs the year, confusingly! 
    StormYear = r.season 


    ## not sure if this is even used
    dVdt = np.full((d1, d2), np.nan)
    dPdt = np.full((d1, d2), np.nan)
    dVdt[2:, :] = StormMwspd[2:, :] - StormMwspd[:-2, :]
    ## in readbst, this is also StormMwspd
    dPdt[1:, :] = StormMwspd[1:,:] - StormMwspd[:-1, :]
    ## thought it should be pressure?
    ## but i don't think this is even used in CHAZ
    #dPdt[1:, :] = StormMslp[1:, :] - StormMslp[:-1, :]

    ## conversion, not totally sure about this yet
    Time = _time_to_numeric(dates)

    ## start structure for netcdf output
    data_vars = dict(
        StormId=(("d2",), StormId),
        StormYear=(("d2",), StormYear),
        StormInitMonth=(("d2",), StormInitMonth),
        StormLon=(("d1", "d2"), StormLon),
        StormLat=(("d1", "d2"), StormLat),
        StormMwspd=(("d1", "d2"), StormMwspd),
        StormMslp=(("d1", "d2"), StormMslp),
        Time=(("d1", "d2"), Time),
        trSpeed=(("d1", "d2"), trSpeed),
        trDir=(("d1", "d2"), trDir),
        dVdt=(("d1", "d2"), dVdt),
        dPdt=(("d1", "d2"), dPdt),
        basin=(("d2"), basin)
    )

    ## add empty placeholders for the predictors
    for f in BT_FIELDS:
        data_vars[f] = (("d1", "d2"), np.full((d1, d2), np.nan))

    if 'global' in basinName:
        data_vars['errors'] = (("d1", "d2"), np.full((d1, d2), np.nan))
        
        ## in original, but not used/necessary in CHAZ, neglecting
        #d3 = 10
        #data_vars['predictors'] = (("d1", "d2", "d3"), np.full((d1, d2, d3), np.nan))
        #data_vars['predictand'] = (("d1", "d2"), np.full((d1, d2), np.nan))
        ## also dropped trueY

                                      
    ds_out = xr.Dataset(data_vars=data_vars)
    #assert len(ds_out.data_vars) == 36, f"expected 36 variables, got {len(ds_out.data_vars)}"
    

    ## add attrs for additional information
    ds_out.attrs['d1'] = ("timestep within storm, 6-hourly, padded with NaN "
                           "beyond each storm's actual length (length = the "
                           "longest-lived storm in this basin)")
    ds_out.attrs['d2'] = f"storm index for {basinName} basin, ({d2} storms)"

    long_names = {
        'StormId': 'IBTrACS season storm number',
        'StormYear': 'season (year) of TC',
        'StormInitMonth': 'month of the TC\'s first recorded position',
        'StormLon': 'center longitude',
        'StormLat': 'center latitude',
        'StormMwspd': 'maximum sustained wind speed',
        'StormMslp': 'minimum sea level pressure',
        'Time': 'days since 1800-01-01',
        'trSpeed': 'storm translation speed',
        'trDir': 'storm translation direction',
        'dVdt': 'wind speed tendency (StormMwspd[t] - StormMwspd[t-2])',
        'dPdt': 'pressure tendency (StormMslp[t] - StormMslp[t-1])',
    }
    units = {
        'StormLon': 'degrees_east', 'StormLat': 'degrees_north',
        'StormMwspd': 'kt', 'StormMslp': 'hPa', #'Time': 'days', ## can't add days because importing will try to interpret, oops
        'trSpeed': 'kt', 'trDir': 'radians', 'dVdt': 'kt', 'dPdt': 'hPa',
    }

    ## update attrs to assert longitude range
    ds_out["StormLon"].attrs.update(valid_min=0.0, valid_max=360.0)

    for name, desc in long_names.items():
        ds_out[name].attrs['long_name'] = desc
    for name, u in units.items():
        ds_out[name].attrs['units'] = u
    
    
    ds_out.to_netcdf(outFileName)
    print(f"wrote {outFileName}  (d1={d1}, d2={d2}, {len(ds_out.data_vars)} variables)")
    return ds_out
 
 
def build_all_basins(ncFileName, basins, outdir):
    for b in basins:
        print(f'starting {b}')
        outFileName = f"{outdir.rstrip('/')}/bt_{b}.nc"
        build_bt(ncFileName, b, outFileName)
 
 
if __name__ == "__main__":
    NC_FILE = '/xpt/taroko.local/data0/clee/bt/IBTrACS.ALL.v04r00.nc'
    #NC_FILE = "/home/miriamn/CHAZ/CHAZvectorized/input/IBTrACS.ALL.v04r01.nc"
    OUTDIR = gv.ipath
    #OUTDIR = "/home/miriamn/CHAZ/CHAZvectorized/input/updating/1981-2012/"
    #build_all_basins(NC_FILE, ['global'], OUTDIR)
    build_all_basins(NC_FILE, ['atl', 'enp', 'wnp', 'sh', 'ni', 'global'], OUTDIR)
    #build_all_basins(NC_FILE, ['atl'], OUTDIR)


starting atl
wrote input/updating/updated-error/bt_atl.nc  (d1=180, d2=2314, 38 variables)
starting enp
wrote input/updating/updated-error/bt_enp.nc  (d1=180, d2=1307, 38 variables)
starting wnp
wrote input/updating/updated-error/bt_wnp.nc  (d1=180, d2=4200, 38 variables)
starting sh
wrote input/updating/updated-error/bt_sh.nc  (d1=180, d2=4165, 38 variables)
starting ni
wrote input/updating/updated-error/bt_ni.nc  (d1=180, d2=1637, 38 variables)
starting global
wrote input/updating/updated-error/bt_global.nc  (d1=180, d2=13623, 39 variables)


Using the IBTrACS file at `'/xpt/taroko.local/data0/clee/bt/IBTrACS.ALL.v04r00.nc'`, 
Update:
    bt_*.StormYear ranges from 1851-2022
Original 
    bt_*.StormYear from 1851-2012

The IBTrACS file has storm dates from 1842 to 2022

In [5]:
import xarray as xr
import numpy as np

from netCDF4 import Dataset


In [6]:
IB_taroko = xr.open_dataset("/xpt/taroko.local/data0/clee/bt/IBTrACS.ALL.v04r00.nc").load()
IB_new = xr.open_dataset("/home/miriamn/CHAZ/CHAZvectorized/input/IBTrACS.ALL.v04r01.nc").load()

## Questions, why is the minimum longitude in both of these -358.5 when the attributes say it should be -180 to 180? 

In [7]:
IB_taroko.usa_lon

<xarray.DataArray 'usa_lon' (storm: 13671, date_time: 360)> Size: 20MB
array([[      nan,       nan,       nan, ...,       nan,       nan,
              nan],
       [      nan,       nan,       nan, ...,       nan,       nan,
              nan],
       [      nan,       nan,       nan, ...,       nan,       nan,
              nan],
       ...,
       [129.2    , 128.95828, 128.6    , ...,       nan,       nan,
              nan],
       [ 88.4    ,  88.60222,  88.8    , ...,       nan,       nan,
              nan],
       [130.7    , 130.25473, 130.     , ...,       nan,       nan,
              nan]], shape=(13671, 360), dtype=float32)
Coordinates:
    time     (storm, date_time) datetime64[ns] 39MB 1842-10-25T03:00:00.00004...
    lat      (storm, date_time) float32 20MB 10.9 10.87 10.84 ... nan nan nan
    lon      (storm, date_time) float32 20MB 80.3 79.83 79.35 ... nan nan nan
Dimensions without coordinates: storm, date_time
Attributes:
    long_name:              Longitude (U.S. Agency)
    units:                  degrees_east
    coverage_content_type:  coordinate

In [8]:
IB_taroko.lon.max()

<xarray.DataArray 'lon' ()> Size: 4B
array(266.9, dtype=float32)
Attributes:
    long_name:              longitude
    standard_name:          longitude
    units:                  degrees_east
    description:            This is merged position based on the position(s) ...
    Note:                   Variable:lon can be missing since the tracks are ...
    coverage_content_type:  coordinate

In [9]:
bt_atl_orig = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/original/bt_atl.nc').load()
bt_atl_updt = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/updating/1981-2012/bt_atl.nc').load()
#bt_atl_updt_long = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/updating/bad-1981-2025/bt_atl.nc').load()

FileNotFoundError: [Errno 2] No such file or directory: '/home/miriamn/CHAZ/CHAZvectorized/input/updating/1981-2012/bt_atl.nc'

In [ ]:
bt_atl_orig

<xarray.Dataset>
Dimensions:         (d1: 132, d2: 1736)
Dimensions without coordinates: d1, d2
Data variables: (12/36)
    PIslp           (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    PIslpMean       (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    PIwspd          (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    PIwspdMean      (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    StormId         (d2) float64 1.0 2.0 3.0 4.0 5.0 ... 16.0 17.0 18.0 19.0
    StormInitMonth  (d2) float64 6.0 7.0 7.0 8.0 9.0 ... 10.0 10.0 10.0 10.0
    ...              ...
    landmaskMean    (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    rh              (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    rh500_300       (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    rhMean          (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    trDir           (d1, d2) float64 3.142 0.0 0.0 2.944 0.0 ... nan nan nan 0.0
    trSpeed         (d1, d2) float64 2.045 0.0 0.0 5.746 0.0 ... nan nan nan 0.0

In [ ]:
bt_atl_updt

<xarray.Dataset>
Dimensions:         (d1: 180, d2: 2314)
Dimensions without coordinates: d1, d2
Data variables: (12/38)
    StormId         (d2) float64 5.0 6.0 7.0 8.0 9.0 ... 68.0 76.0 77.0 79.0
    StormYear       (d2) float64 1.851e+03 1.851e+03 ... 2.022e+03 2.022e+03
    StormInitMonth  (d2) float64 6.0 6.0 7.0 7.0 8.0 ... 9.0 9.0 10.0 10.0 10.0
    StormLon        (d1, d2) float64 nan nan -97.6 -60.0 nan ... nan nan nan nan
    StormLat        (d1, d2) float64 nan nan 22.2 12.0 nan ... nan nan nan nan
    StormMwspd      (d1, d2) float64 nan nan 80.0 50.0 nan ... nan nan nan nan
    ...              ...
    dThetaEMean     (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    dThetaE         (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    dThetaEsMean    (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    dThetaEs        (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    landmask        (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
    landmaskMean    (d1, d2) float64 nan nan nan nan nan ... nan nan nan nan nan
Attributes:
    d1:       timestep within storm, 6-hourly, padded with NaN beyond each st...
    d2:       storm index for atl basin, (2314 storms)

In [ ]:
np.unique(bt_atl_orig.StormYear)

array([1851., 1852., 1853., 1854., 1855., 1856., 1857., 1858., 1859.,
       1860., 1861., 1862., 1863., 1864., 1865., 1866., 1867., 1868.,
       1869., 1870., 1871., 1872., 1873., 1874., 1875., 1876., 1877.,
       1878., 1879., 1880., 1881., 1882., 1883., 1884., 1885., 1886.,
       1887., 1888., 1889., 1890., 1891., 1892., 1893., 1894., 1895.,
       1896., 1897., 1898., 1899., 1900., 1901., 1902., 1903., 1904.,
       1905., 1906., 1907., 1908., 1909., 1910., 1911., 1912., 1913.,
       1914., 1915., 1916., 1917., 1918., 1919., 1920., 1921., 1922.,
       1923., 1924., 1925., 1926., 1927., 1928., 1929., 1930., 1931.,
       1932., 1933., 1934., 1935., 1936., 1937., 1938., 1939., 1940.,
       1941., 1942., 1943., 1944., 1945., 1946., 1947., 1948., 1949.,
       1950., 1951., 1952., 1953., 1954., 1955., 1956., 1957., 1958.,
       1959., 1960., 1961., 1962., 1963., 1964., 1965., 1966., 1967.,
       1968., 1969., 1970., 1971., 1972., 1973., 1974., 1975., 1976.,
       1977., 1978.,

In [ ]:
print('Original bt_atl.nc')
print(f'Number of unique StormId: {len(np.unique(bt_atl_orig.StormId))}')

Original bt_atl.nc
Number of unique StormId: 31


In [ ]:
np.unique(bt_atl_updt.StormId)

print('IBTrACS on Taroko')
print(f'Number of unique StormId: {len(np.unique(bt_atl_updt.StormId))}')

IBTrACS on Taroko
Number of unique StormId: 138


In [ ]:
print('Most recent IBTrACS')
print(f'Number of unique StormId: {len(np.unique(bt_atl_updt_long.StormId))}')

Most recent IBTrACS
Number of unique StormId: 127


In [ ]:
bt_atl_updt_long.Time

<xarray.DataArray 'Time' (d1: 180, d2: 2294)> Size: 3MB
array([[18800.5 , 18806.5 , 18812.5 , ..., 82780.  , 82784.75, 82804.5 ],
       [18800.75, 18806.75,     0.  , ..., 82780.25, 82785.  , 82804.75],
       [18801.  , 18807.  ,     0.  , ..., 82780.5 , 82785.25, 82805.  ],
       ...,
       [    0.  ,     0.  ,     0.  , ...,     0.  ,     0.  ,     0.  ],
       [    0.  ,     0.  ,     0.  , ...,     0.  ,     0.  ,     0.  ],
       [    0.  ,     0.  ,     0.  , ...,     0.  ,     0.  ,     0.  ]],
      shape=(180, 2294))
Dimensions without coordinates: d1, d2

In [ ]:
len(np.unique(bt_atl_updt.StormYear))

172

In [ ]:
bt_atl_orig.Time

<xarray.DataArray 'Time' (d1: 132, d2: 1736)> Size: 2MB
array([[18802.  , 18812.5 , 18817.5 , ..., 77716.75, 77725.75, 77725.75],
       [18802.25,     0.  ,     0.  , ..., 77717.  , 77726.  , 77726.  ],
       [18802.5 ,     0.  ,     0.  , ..., 77717.25, 77726.25, 77726.25],
       ...,
       [    0.  ,     0.  ,     0.  , ...,     0.  ,     0.  ,     0.  ],
       [    0.  ,     0.  ,     0.  , ...,     0.  ,     0.  ,     0.  ],
       [    0.  ,     0.  ,     0.  , ...,     0.  ,     0.  ,     0.  ]],
      shape=(132, 1736))
Dimensions without coordinates: d1, d2

In [ ]:
# IBTrACS_fname = '/xpt/taroko.local/data0/clee/bt/IBTrACS.ALL.v04r00.nc'
# r = read_ibtracs_v4_xr(IBTrACS_fname , ['atl'])


In [ ]:
# bt_global_original = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/original/bt_global_predictors.nc').load()
# bt_global_update = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/bt_global_predictors.nc').load()

In [ ]:
np.unique(bt_global_original.StormYear)

NameError: name 'bt_global_original' is not defined

In [ ]:
np.unique(bt_global_update.StormYear)

In [ ]:
bt_atl_orig = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/original/bt_atl.nc').load()
bt_atl_update = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/bt_atl.nc').load()

In [ ]:
bt_atl_orig

In [ ]:
bt_atl_update

In [ ]:
bt_global_original

In [ ]:
bt_global_update

In [ ]:
# ncFileName = "/home/miriamn/CHAZ/CHAZvectorized/input/IBTrACS.ALL.v04r01.nc"
# basinName = 'global'
# r = read_ibtracs_v4_xr(ncFileName, [basinName])

In [ ]:
# #IBTrACS_fname = '/xpt/taroko.local/data0/clee/bt/IBTrACS.ALL.v04r00.nc'
# IBTrACS_fname = "/home/miriamn/CHAZ/CHAZvectorized/input/IBTrACS.ALL.v04r01.nc"
# tracs = xr.open_dataset(IBTrACS_fname).load()

In [ ]:
# bt_atl_update = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/updating/bt_atl.nc').load()
# bt_atl_original = bt_atl

In [ ]:
# np.sort(list(bt_atl_update))

In [ ]:
# np.sort(list(bt_atl_original))

In [ ]:
# #bt_atl_update.Time.where(bt_atl_update.Time > 0).min()
# bt_atl_update.Time

In [ ]:
# #bt_atl_original.Time.where(bt_atl_original.Time > 0).min()
# bt_atl_original.Time

In [ ]:
# bt_global_update = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/updating/bt_global.nc').load()
# bt_global_original = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/bt_global_predictors.nc').load()

In [ ]:
# np.sort(list(bt_global_update))

In [ ]:
# np.sort(list(bt_global_original))

# ## add basin, errors (need data?), predictand, predictors, rh500_300Mean
# ## don't see predictand and predictors but do see errors? 

In [ ]:
# bt_global_update

In [ ]:
# bt_global_original

In [ ]:
# bt_enp_update = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/bt_enp.nc').load()
# bt_enp_original = xr.open_dataset('/home/miriamn/CHAZ/CHAZvectorized/input/original/bt_enp.nc').load()

In [ ]:
# bt_enp_update.UShear.max(skipna = True)

In [ ]:
# bt_enp_original.UShear.max(skipna = True)

In [ ]:
# np.nanmax(bt_enp_original.UShear)

In [ ]:
# IBTrACS_fname = '/xpt/taroko.local/data0/clee/bt/IBTrACS.ALL.v04r00.nc'

# #r = read_ibtracs_v4_xr(IBTrACS_fname , ['atl'])


In [ ]:
# ibtracs = xr.open_dataset(IBTrACS_fname).load()
# ibtracs

In [ ]:
# ibtracs

In [ ]:
# ds = xr.open_dataset(IBTrACS_fname)

In [ ]:
    # ds_out = xr.Dataset(
    #     data_vars=dict(
    #         StormId=(("d2",), StormId),
    #         StormYear=(("d2",), StormYear),
    #         StormInitMonth=(("d2",), StormInitMonth),
    #         StormLon=(("d1", "d2"), StormLon),
    #         StormLat=(("d1", "d2"), StormLat),
    #         StormMwspd=(("d1", "d2"), StormMwspd),
    #         StormMslp=(("d1", "d2"), StormMslp),
    #         StormYYYY=(("d1", "d2"), StormYYYY),
    #         StormMM=(("d1", "d2"), StormMM),
    #         StormDD=(("d1", "d2"), StormDD),
    #         StormHH=(("d1", "d2"), StormHH),
    #         trSpeed=(("d1", "d2"), trSpeed),
    #         trDir=(("d1", "d2"), trDir),
    #         dist2land=(("d1", "d2"), dist2land),
    #         dVdt=(("d1", "d2"), dVdt),
    #         dPdt=(("d1", "d2"), dPdt),
    #     )
    # )
    # ds_out.to_netcdf(outFileName)
    # print(f"saved {outFileName}  (d1={d1}, d2={d2})")
    # return ds_out